In [8]:
import pandas as pd

df = pd.read_csv("data_raw/antibiotics_raw_data.csv")

duplicates = df[df.duplicated(subset=['Area Code', 'Indicator ID', 'Time period'], keep=False)]

print(duplicates[['Area Name', 'Indicator ID', 'Time period', 'Value']].head(20))

    Area Name  Indicator ID Time period      Value
0     England         92163     2015 Q1  164.86879
1     England         92163     2015 Q1  164.86879
102   England         92163     2015 Q2  135.01398
103   England         92163     2015 Q2  135.01398
204   England         92163     2015 Q3  126.56557
205   England         92163     2015 Q3  126.56557
306   England         92163     2015 Q4  148.44063
307   England         92163     2015 Q4  148.44063
408   England         92163     2016 Q1  151.88916
409   England         92163     2016 Q1  151.88916
510   England         92163     2016 Q2  132.49615
511   England         92163     2016 Q2  132.49615
612   England         92163     2016 Q3  125.20323
613   England         92163     2016 Q3  125.20323
714   England         92163     2016 Q4  149.60376
715   England         92163     2016 Q4  149.60376
816   England         92163     2017 Q1  145.49603
817   England         92163     2017 Q1  145.49603
918   England         92163    

In [9]:
df['Area Type'].unique()

array(['England', 'ICB sub-locations'], dtype=object)

In [12]:
df_filtered = df[df['Area Type'] == 'ICB sub-locations'].copy()

In [17]:
duplicate_after = df_filtered[df_filtered.duplicated(subset=['Area Code','Indicator ID','Time period'], keep = False)]
len(duplicate_after)

0

In [18]:
len(df_filtered)

9064

In [19]:
df_pivot = df_filtered.pivot(
    index=['Area Code', 'Area Name'], 
    columns=['Indicator ID', 'Time period'], 
    values='Value'
)
df_pivot.shape

(106, 88)

In [21]:
print(df_pivot.columns[:5])

MultiIndex([(92163, '2015 Q1'),
            (92163, '2015 Q2'),
            (92163, '2015 Q3'),
            (92163, '2015 Q4'),
            (92163, '2016 Q1')],
           names=['Indicator ID', 'Time period'])


In [22]:
df_pivot.columns = [f"Ind_{int(i)}_{t.replace(' ', '_')}" for i, t in df_pivot.columns]

df_master = df_pivot.reset_index()

print(df_master.columns[:7].tolist())

['Area Code', 'Area Name', 'Ind_92163_2015_Q1', 'Ind_92163_2015_Q2', 'Ind_92163_2015_Q3', 'Ind_92163_2015_Q4', 'Ind_92163_2016_Q1']


In [26]:

current_year = ['2024_Q3', '2024_Q4', '2025_Q1', '2025_Q2']
previous_year = ['2023_Q3', '2023_Q4', '2024_Q1', '2024_Q2']

qty_current_cols = [f"Ind_92163_{q}" for q in current_year]
qlty_current_cols = [f"Ind_92167_{q}" for q in current_year]

df_master['Qty_Mean_Current'] = df_master[qty_current_cols].mean(axis=1)
df_master['Quality_Mean_Current'] = df_master[qlty_current_cols].mean(axis=1)

df_master['Seasonality'] = (df_master['Ind_92163_2024_Q4'] + df_master['Ind_92163_2025_Q1']) / \
                           (df_master['Ind_92163_2024_Q3'] + df_master['Ind_92163_2025_Q2'])

qty_prev_cols = [f"Ind_92163_{q}" for q in previous_year]
df_master['Qty_Prev_Mean'] = df_master[qty_prev_cols].mean(axis=1)
df_master['Quantity_Trend'] = ((df_master['Qty_Mean_Current'] - df_master['Qty_Prev_Mean']) / 
                               df_master['Qty_Prev_Mean']) * 100
df_final = df_master[['Area Code', 'Area Name', 'Qty_Mean_Current', 'Quality_Mean_Current', 'Seasonality', 'Quantity_Trend']]

print(df_final.head())

   Area Code                                  Area Name  Qty_Mean_Current  \
0  E38000006                 South Yorkshire ICB - 02P         126.375080   
1  E38000007             Mid and South Essex ICB - 99E         115.499282   
2  E38000008  Nottingham and Nottinghamshire ICB - 02Q         133.863192   
3  E38000014    Lancashire and South Cumbria ICB - 00Q         119.965815   
4  E38000015    Lancashire and South Cumbria ICB - 00R         119.021610   

   Quality_Mean_Current  Seasonality  Quantity_Trend  
0                5.2700     1.156840       -7.183323  
1                7.6750     1.267248      -10.736861  
2                6.8825     1.153237      -10.075656  
3                5.7900     1.168158      -12.869691  
4                8.5850     1.134243      -12.737459  


In [28]:
df_final.to_csv ('data_raw/clustering_input.csv',index=False)